# Worklet 9 — Placement Readiness & Career Intelligence Portal
## Lab 08: Parallel swarm, partial results and reconciliation

**Team:** Vijay Kumar — 01FE23BCI083; Rishi Kulkarni — 01FE23BCI088; Prajwal Hugar — 01FE23BCI075; Rohit Yendegeri — 01FE23BCI001.

**Problem statement:** Help students understand resume/JD gaps and prepare with auditable, placement-officer-reviewed guidance.

| Item | This lab |
| --- | --- |
| Goal | Compare serial/parallel student processing, bounded retries and identity-aware merge. |
| Inputs | Two synthetic students plus an intentionally unavailable student. |
| Deliverable | Executable demonstration, assertions and recorded output |
| Acceptance | Equal serial/parallel quality, bounded retries, explicit missing coverage and no duplicate student totals. |

Run top to bottom from this repository. The notebook imports `lab_support.py` and existing backend modules; it is independent of other notebooks, but is not a standalone Colab upload. All examples use synthetic data.


### 1. Environment and scope

Only local code and disposable fixtures are used. Dependencies: Python 3.11+ and the repository backend requirements. No secrets, real student records, or reference-notebook outputs are copied.


In [1]:
from pathlib import Path
import sys
folder=Path.cwd()
if not (folder/'lab_support.py').exists():folder=folder/'Assignments'
assert (folder/'lab_support.py').exists(), 'Run from the repository root or Assignments directory'
sys.path.insert(0,str(folder.resolve()))
from lab_support import *
print('Worklet 9: offline synthetic demonstration; no Gemini API calls')


Worklet 9: offline synthetic demonstration; no Gemini API calls


### 2. Worklet-specific implementation and experiment

The following code executes the stated scenario and deliberately exercises rejection paths where relevant.


In [2]:
started=time.perf_counter();serial=[report(s) for s in STUDENTS];serial_time=time.perf_counter()-started
started=time.perf_counter();parallel=swarm(list(STUDENTS),report);parallel_time=time.perf_counter()-started
assert [r['coverage'] for r in serial]==[r['report']['coverage'] for r in parallel['results']]
calls={}
def flaky(student):
 calls[student]=calls.get(student,0)+1
 if student=='MISSING' or (student=='DEMO002' and calls[student]==1):raise ValueError('Injected unavailable evidence')
 return report(student)
partial=swarm(['DEMO001','DEMO002','MISSING'],flaky)
assert partial['missing']==['MISSING'] and calls['DEMO002']==2 and calls['MISSING']==2
assert partial['coverage']==2/3
merged,conflicts=merge_versions([{'student':'DEMO001','version':1},{'student':'DEMO001','version':2},{'student':'DEMO002','version':1}])
assert len(merged)==2 and conflicts==['DEMO001'] and merged[0]['version']==2
show({'sequential_seconds':serial_time,'parallel_seconds':parallel_time,'partial_coverage':partial['coverage'],'missing':partial['missing'],'merged':merged,'conflicts':conflicts})


{
  "sequential_seconds": 0.0332937000785023,
  "parallel_seconds": 0.009735299972817302,
  "partial_coverage": 0.6666666666666666,
  "missing": [
    "MISSING"
  ],
  "merged": [
    {
      "student": "DEMO001",
      "version": 2
    },
    {
      "student": "DEMO002",
      "version": 1
    }
  ],
  "conflicts": [
    "DEMO001"
  ]
}


### 3. Acceptance checks against the actual portal

The demonstration asserts its own contract above. This additional test calls the project implementation using disposable fixtures. Implementation references: `backend/runtime_service.py; backend/benchmark_swarm.py`.


In [3]:
regression_result=regressions('test_runtime.RuntimeTests.test_swarm_matches_single_run_quality_and_bounds_workers')
show(regression_result)


.
----------------------------------------------------------------------
Ran 1 test in 2.225s

OK

{
  "tests": 1,
  "failures": 0,
  "errors": 0
}


### 4. Results, reflection and limits

Timing is measured, not assumed faster. Identity reconciliation and per-worker retry are notebook extensions; the portal batch has independent student outcomes, two workers and runtime retry support.

The output above is generated by execution, not an expected-output transcription. If any assertion or regression fails, this notebook is not accepted.

### 5. Deliverables and next step

- This executed notebook and its assertion output.
- Shared original teaching helpers in `lab_support.py`.
- Relevant production code and regression mapping above.
- Full sequence and provenance in `Assignments/README.md`.

Continue to Lab 9 to build on this contract.
